In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Load data
df = pd.read_csv('Forest_data.csv', encoding='latin-1')

# Country lists
ape_countries = [
    'Democratic Republic of the Congo', 'Cameroon', 'Gabon',
    'Republic of the Congo', 'Central African Republic', 'Nigeria',
    'Uganda', 'Rwanda', 'Burundi', 'Tanzania',
    'Indonesia', 'Malaysia'
]

africa_countries = [
    'Democratic Republic of the Congo', 'Cameroon', 'Gabon',
    'Republic of the Congo', 'Central African Republic', 'Nigeria',
    'Uganda', 'Rwanda', 'Burundi', 'Tanzania'
]

asia_countries = [
    'Indonesia', 'Malaysia'
]

# Filter to threshold 10 and ape countries
df_filtered = df[
    (df['threshold'] == 10) &
    (df['country'].isin(ape_countries))
]

# Select country, area and annual loss columns
loss_columns = [col for col in df.columns if col.startswith('tc_loss_ha_')]
df_loss = df_filtered[['country', 'area_ha', 'extent_2000_ha'] + loss_columns]

# Reshape from wide to long
df_long = df_loss.melt(
    id_vars=['country', 'area_ha', 'extent_2000_ha'],
    value_vars=loss_columns,
    var_name='year',
    value_name='forest_loss_ha'
)

# Clean year column
df_long['year'] = df_long['year'].str.replace('tc_loss_ha_', '').astype(int)

# Calculate loss as percentage of forest extent
df_long['loss_percent'] = (df_long['forest_loss_ha'] / df_long['extent_2000_ha']) * 100

# Build figure
fig = px.line(
    df_long,
    x='year',
    y='loss_percent',
    color='country',
    title="<b>Forest loss by country, 2001-2025</b><br><sup><i>Expressed as a percentage of country's forested area</i></sup>",
    labels={'loss_percent': 'Forest cover loss (%)', 'year': 'Year'},
    template='simple_white',
    custom_data=['country', 'loss_percent']
)

# Tooltip
fig.update_traces(
    hovertemplate=
        '<b>%{customdata[0]}</b><br>' +
        'Year: %{x}<br>' +
        'Forest cover loss: %{customdata[1]:.3f}%' +
        '<extra></extra>',
    mode='lines+markers',
    marker=dict(size=5)
)

# Y axis styling
fig.update_yaxes(
    gridcolor='lightgrey',
    griddash='dash',
    gridwidth=0.5
)

# X axis range
fig.update_xaxes(range=[2001, 2025])

# Annotations
fig.add_annotation(
    text="Click to isolate a country  •  Double-click to hide a country",
    xref='paper', yref='paper',
    x=0.5, y=-0.20,
    showarrow=False,
    font=dict(size=12, color='grey'),
    xanchor='center'
)

fig.add_annotation(
    text="Source: Global Forest Watch, Hansen et al.",
    xref='paper', yref='paper',
    x=0, y=-0.4,
    showarrow=False,
    font=dict(size=12, color='grey'),
    xanchor='left'
)

# Layout
fig.update_layout(
    modebar_remove=['pan', 'select', 'lasso'],
    height=650,
    showlegend=True,
    legend=dict(
        font=dict(size=11),
        itemclick='toggleothers',
        itemdoubleclick='toggle',
        title=dict(text='<b>Country</b>', font=dict(size=11)),
        orientation='h',
        x=0.4,
        y=-0.2,
        xanchor='center'
    ),
    margin=dict(t=75, b=170),
    font_family='Roboto',
    title_font_size=20,
    hovermode='closest',
    updatemenus=[
        dict(
            type='buttons',
            direction='right',
            x=0.7,
            y=1.08,
            xanchor='left',
            buttons=[
                dict(
                    label='All countries',
                    method='update',
                    args=[
                        {
                            'visible': [True] * len(fig.data),
                            'hovertemplate': (
                                '<b>%{customdata[0]}</b><br>' +
                                'Year: %{x}<br>' +
                                'Forest cover loss: %{customdata[1]:.3f}%' +
                                '<extra></extra>'
                            )
                        },
                        {'hovermode': 'closest'}
                    ]
                ),
                dict(
                    label='Africa',
                    method='update',
                    args=[
                        {
                            'visible': [trace.name in africa_countries for trace in fig.data],
                            'hovertemplate': '%{y:.3f}%'
                        },
                        {
                            'hovermode': 'x unified',
                            'hoversort': 'value descending',
                            'hoverlabel.font.size': 14,
                            'hoverlabel.align': 'left'
                        }
                    ]
                ),
                dict(
                    label='Asia',
                    method='update',
                    args=[
                        {
                            'visible': [trace.name in asia_countries for trace in fig.data],
                            'hovertemplate': '%{y:.3f}%'
                        },
                        {
                            'hovermode': 'x unified',
                            'hoverlabel.font.size': 14,
                            'hoverlabel.align': 'left'
                        }
                    ]
                ),
            ]
        )
    ]
)

fig.show()

In [ ]:
print(ape_ranges['country'].unique())

In [ ]:
import pandas as pd

# Load your dataframe - adjust path to match your setup
df = pd.read_csv(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Forest_data.csv', encoding='latin-1')

# Filter to 10% canopy threshold if multiple thresholds present
df = df[df['threshold'] == 10]
region_map = {
    # Africa
    'Democratic Republic of the Congo': 'Africa',
    'Cameroon': 'Africa',
    'Gabon': 'Africa',
    'Republic of the Congo': 'Africa',
    'Central African Republic': 'Africa',
    'Nigeria': 'Africa',
    'Uganda': 'Africa',
    'Rwanda': 'Africa',
    'Burundi': 'Africa',
    'Tanzania': 'Africa',
    # Asia
    'Indonesia': 'Asia',
    'Malaysia': 'Asia',
}

df['region'] = df['country'].map(region_map)
# Define loss columns
loss_cols = [f'tc_loss_ha_{y}' for y in range(2001, 2026)]

# Total across all countries in analysis
total_loss_ha = df[loss_cols].sum().sum()

# By region
africa_loss = df[df['region'] == 'Africa'][loss_cols].sum().sum()
asia_loss = df[df['region'] == 'Asia'][loss_cols].sum().sum()

# Convert to million hectares
print(f"Total:  {total_loss_ha / 1_000_000:.2f} Mha  ({total_loss_ha:,.0f} ha)")
print(f"Africa: {africa_loss / 1_000_000:.2f} Mha  ({africa_loss:,.0f} ha)")
print(f"Asia:   {asia_loss / 1_000_000:.2f} Mha  ({asia_loss:,.0f} ha)")

In [ ]:
region_map = {
    'Democratic Republic of the Congo': 'Africa',
    'Cameroon': 'Africa',
    'Gabon': 'Africa',
    'Republic of the Congo': 'Africa',
    'Central African Republic': 'Africa',
    'Nigeria': 'Africa',
    'Uganda': 'Africa',
    'Rwanda': 'Africa',
    'Burundi': 'Africa',
    'Tanzania': 'Africa',
    'Indonesia': 'Asia',
    'Malaysia': 'Asia',
}

df_filtered['region'] = df_filtered['country'].map(region_map)

for region in ['Africa', 'Asia']:
    subset = df_filtered[df_filtered['region'] == region]
    total_loss = subset[loss_columns].sum().sum()
    total_extent = subset['extent_2000_ha'].sum()
    pct = (total_loss / total_extent) * 100
    print(f"{region}: {total_loss/1_000_000:.2f} Mha lost of {total_extent/1_000_000:.2f} Mha extent ({pct:.1f}%)")

In [ ]:
extent_by_country = df_filtered.groupby('country')['extent_2000_ha'].sum().sort_values(ascending=False)
print(extent_by_country.apply(lambda x: f"{x/1_000_000:.2f} Mha"))

In [ ]:

import pandas as pd
import plotly.graph_objects as go

# Load data
df = pd.read_csv(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Forest_data.csv', encoding='latin-1')

ape_countries = [
    'Democratic Republic of the Congo', 'Cameroon', 'Gabon',
    'Republic of the Congo', 'Central African Republic', 'Nigeria',
    'Uganda', 'Rwanda', 'Burundi', 'Tanzania',
    'Indonesia', 'Malaysia'
]

df_filtered = df[(df['threshold'] == 10) & (df['country'].isin(ape_countries))]
loss_columns = [col for col in df.columns if col.startswith('tc_loss_ha_')]
df_loss = df_filtered[['country', 'extent_2000_ha'] + loss_columns]

df_long = df_loss.melt(
    id_vars=['country', 'extent_2000_ha'],
    value_vars=loss_columns,
    var_name='year',
    value_name='forest_loss_ha'
)
df_long['year'] = df_long['year'].str.replace('tc_loss_ha_', '').astype(int)
df_long['loss_percent'] = (df_long['forest_loss_ha'] / df_long['extent_2000_ha']) * 100

# Heatmap data
df_heatmap = df_long.pivot(index='country', columns='year', values='loss_percent')
country_names = {
    'Democratic Republic of the Congo': 'DR Congo',
    'Republic of the Congo': 'Rep. of Congo',
    'Central African Republic': 'Central African Rep.'
}
# Cumulative raw hectares for bar chart
cumulative = df_long.groupby('country', as_index=False)['forest_loss_ha'].sum()
cumulative = cumulative.rename(columns={'forest_loss_ha': 'cumulative_loss_ha'})
cumulative['cumulative_loss_mha'] = cumulative['cumulative_loss_ha'] / 1_000_000
cumulative['country'] = cumulative['country'].replace(country_names)

fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(
    x=cumulative["cumulative_loss_mha"],
    y=cumulative["country"],
    orientation='h',
    marker=dict(
        color=cumulative["cumulative_loss_mha"],
        colorscale='RdYlGn_r',
        cmin=0,
        cmax=15,
        showscale=False
    ),
    hovertemplate='<b>%{y}</b><br>Cumulative tree cover loss: %{x:.2f} Mha<extra></extra>'
))
fig_bar.update_layout(
    title=dict(
        text='<b>Tree cover loss by country, 2001–2025</b><br>'
             '<sup><i>Cumulative loss in million hectares</i></sup>',
        x=0.02
    ),
    template='simple_white',
    font_family='Roboto',
    height=600,
    margin=dict(t=60, b=130),
    xaxis=dict(
        title='<b>Cumulative tree cover loss (Mha)</b>',
        tickfont=dict(size=13),
        fixedrange=True
    ),
    yaxis=dict(
        tickfont=dict(size=13),
        categoryorder='total ascending'
    ),
    modebar_remove=['pan2d', 'select2d', 'lasso2d', 'zoom2d', 'zoomIn2d',
                    'zoomOut2d', 'autoScale2d', 'resetScale2d']
)
fig_bar.add_annotation(
    text="<b>Data source</b>: Hansen et al. (2013), Global Forest Change v GFC-2025-v1.13; Global Forest Watch",
    xref='paper', yref='paper',
    x=-0.14, y=-0.3,
    showarrow=False,
    font=dict(size=12, color='#555555', style='italic'),
    xanchor='left',
    align='left'
)

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Pivot to wide format for heatmap
df_heatmap = df_long.pivot(index='country', columns='year', values='loss_percent')

# Order countries by total loss (highest at top)
df_heatmap['total'] = df_heatmap.sum(axis=1)
df_heatmap = df_heatmap.sort_values('total', ascending=True).drop(columns='total')

fig_heat = go.Figure(data=go.Heatmap(
    z=df_heatmap.values,
    x=df_heatmap.columns.tolist(),
    y=df_heatmap.index.tolist(),
    xgap=1,
    ygap=1,
    zmax=1.2,
    zmin=0,
    colorscale='RdYlGn_r',
    hovertemplate='<b>%{y}</b><br>Year: %{x}<br>Forest loss: %{z:.3f}%<extra></extra>',
    colorbar=dict(
    title=dict(
        text='<b>Forest loss (%)</b>',
        font=dict(size=14,)
    ),
    tickfont=dict(size=11)
)))

fig_heat.update_layout(
    title='<b>Forest loss by country, 2001–2025</b><br><sup><i>Expressed as % of 2000 forest extent within great ape range countries</i></sup>',
    template='simple_white',
    font_family='Roboto',
    title_font_size=18,
    title_x=0.02,
    height=500,
    xaxis=dict(
        title='Year',
        tickmode='array',
        tickvals=[2001, 2005, 2010, 2015, 2020, 2025],
        tickfont=dict(size=14)
    ),
    yaxis=dict(
        tickfont=dict(size=14)
    ),
    margin=dict(t=80, b=80)
)

fig_heat.show()

In [ ]:
print(df_long[df_long['country'] == 'Nigeria'].groupby('year')['loss_percent'].sum().round(3))

In [ ]:
# Calculate total percent loss per country
total_loss = df_long.groupby('country').apply(
    lambda x: (x['forest_loss_ha'].sum() / x['extent_2000_ha'].iloc[0]) * 100
).reset_index()
total_loss.columns = ['Country', 'Total Forest Loss (% of 2000 forest extent)']
total_loss = total_loss.sort_values('Total Forest Loss (% of 2000 forest extent)', ascending=False).reset_index(drop=True)
total_loss['Total Forest Loss (% of 2000 forest extent)'] = total_loss['Total Forest Loss (% of 2000 forest extent)'].apply(lambda x: f"{x:.1f}%")

print(total_loss.to_string(index=False))total_loss.style.set_caption("Total forest cover loss by country, 2001–2025")

In [ ]:
total_loss.style.set_caption("Total forest cover loss by country, 2001–2025")

In [ ]:
african_total = df_long[df_long['country'].isin(africa_countries)]['forest_loss_ha'].sum()
print(f"{african_total:,.0f}")

In [ ]:
african_percent = df_long[df_long['country'].isin(africa_countries)].groupby('country').apply(
    lambda x: (x['forest_loss_ha'].sum() / x['extent_2000_ha'].iloc[0]) * 100
)
print(african_percent)
print(f"\nAverage across African ape countries: {african_percent.mean():.1f}%")
print(f"Total weighted: {african_percent.sum():.1f}%")

In [ ]:
africa_df = df_long[df_long['country'].isin(africa_countries)]
by_country = africa_df.groupby('country')['forest_loss_ha'].sum().sort_values(ascending=False)
print(by_country)
print(f"\nTotal: {by_country.sum():,.0f} ha")

In [ ]:
import geopandas as gpd
print(gpd.__version__)

In [ ]:
import geopandas as gpd

ape_ranges = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Ape_ranges.shp')
protected_areas = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\PAs_within_ape_ranges.shp')
print("Ape ranges CRS:", ape_ranges.crs)
print("Protected areas CRS:", protected_areas.crs)
print("\nApe ranges columns:", ape_ranges.columns.tolist())
print("Protected areas columns:", protected_areas.columns.tolist())
print("\nApe ranges features:", len(ape_ranges))
print("Protected areas features:", len(protected_areas))
print("\nApe ranges preview:")
print(ape_ranges.head())

In [ ]:
print(ape_ranges.columns.tolist())

In [ ]:
import folium
from folium import GeoJson, LayerControl
import geopandas as gpd

# Fix protected areas CRS
protected_areas = protected_areas.set_crs('EPSG:4326')

# Filter to Africa species
africa_species = ['Pan troglodytes','Gorilla gorilla', 'Gorilla beringei', 'Pan paniscus']


# Filter to core range only (excludes West Africa west of 5E)
from shapely.geometry import box
core_bbox = box(5, -15, 50, 15)

ape_africa = ape_ranges[ape_ranges['sci_name'].isin(africa_species)].copy()
ape_africa = ape_africa[ape_africa.intersects(core_bbox)]

pa_africa = protected_areas[protected_areas['sci_name'].isin(africa_species)]

# Colour per species
species_colors = {
    'Gorilla gorilla':    '#797d62',   
    'Gorilla beringei':   '#9b9b7a',   
    'Pan troglodytes':    '#ffcb69',  
    'Pan paniscus':       '#7dcfb6',   
}

species_labels = {
    'Gorilla gorilla':    'Western Gorilla',
    'Gorilla beringei':   'Eastern Gorilla',
    'Pan troglodytes':    'Chimpanzee',
    'Pan paniscus':       'Bonobo',
}

# Build map
m_africa = folium.Map(location=[0, 22], zoom_start=5, tiles=None)
folium.TileLayer('CartoDB positron', control=False).add_to(m_africa)

# Add one layer per species
for species in africa_species:
    species_data = ape_africa[ape_africa['sci_name'] == species]
    color = species_colors[species]
    label = species_labels[species]

    GeoJson(
        species_data,
        name=label,
        style_function=lambda x, c=color: {
            'fillColor': c,
            'color': c,
            'weight': 1,
            'fillOpacity': 0.5
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['sci_name'],
            aliases=['Species:']
        )
    ).add_to(m_africa)

# Protected areas layer
GeoJson(
    pa_africa,
    name='Protected Areas',
    style_function=lambda x: {
        'fillColor': '#64a6bd',
        'color': '#64a6bd',
        'weight': 1,
        'fillOpacity': 0.6
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAME_ENG', 'IUCN_CAT'],
        aliases=['Protected Area:', 'IUCN Category:']
    )
).add_to(m_africa)

LayerControl(collapsed=False, title = 'Select layers', hide_basemap=True).add_to(m_africa)

# Title
title_html = '''
<div style="position: fixed; top: 25px; left: 100px; z-index: 1000; 
background-color: white; 
padding: 10px; 
border-radius: 5px; 
box-shadow: 2px 2px 5px rgba(0,0,0,0.3); 
font-family: Arial;
font-size: 14px;">

<b>Great Ape Ranges and Protected Areas — Africa</b><br>
<span style="color: #797d62">■</span> Western Gorilla &nbsp;
<span style="color: #9b9b7a">■</span> Eastern Gorilla &nbsp;
<span style="color: #ffcb69">■</span> Chimpanzee &nbsp;
<span style="color: #7dcfb6">■</span> Bonobo &nbsp;
<span style="color: #64a6bd">■</span> Protected Areas
</div>
'''
m_africa.get_root().html.add_child(folium.Element(title_html))

css_html = '''
<style>
.leaflet-top.leaflet-right {
    margin-top: 25px;
    margin-right: 100px;
}

.leaflet-control-layers {
    font-family: Arial;
    font-size: 14px;
    border-radius: 5px;
    border: 1px solid #cccccc;
    background-color: white;
}

.leaflet-control-layers-list::before {
    content: 'Layers';
    font-weight: bold;
    font-size: 13px;
    display: block;
    margin-bottom: 8px;
    padding-bottom: 6px;
    border-bottom: 1px solid #eeeeee;
}

.leaflet-control-layers-expanded {
    padding: 12px;
    width: 200px;
}

.leaflet-control-layers label {
    margin-bottom: 6px;
    display: flex;
    align-items: center;
    gap: 6px;
}

.leaflet-control-layers-overlays {
    font-size: 13px;
}

.leaflet-control-layers-toggle {
    background-size: 20px;
}
</style>
'''
m_africa.get_root().html.add_child(folium.Element(css_html))




m_africa.save('africa_map.html')
print("Africa map saved")

In [ ]:
import webbrowser
webbrowser.open('africa_map.html')

In [ ]:
import folium
from folium import GeoJson, LayerControl
import geopandas as gpd

#Import shape files
ape_ranges = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Ape_ranges.shp')
protected_areas = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\PAs_within_ape_ranges.shp')

# Fix protected areas CRS
protected_areas = protected_areas.set_crs('EPSG:4326')

In [ ]:
import geopandas as gpd

# Load shapefiles
ape_ranges = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Ape_ranges.shp')
protected_areas = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\PAs_within_ape_ranges.shp')

# Fix CRS - project to equal area for accurate area calculations
# EPSG:6933 is a good equal area projection for global data
ape_ranges_proj = ape_ranges.to_crs('EPSG:6933')
protected_areas_proj = protected_areas.set_crs('EPSG:4326').to_crs('EPSG:6933')

# Species labels
species_labels = {
    'Gorilla gorilla':  'Gorilla',
    'Gorilla beringei': 'Gorilla',
    'Pan troglodytes':  'Chimpanzee',
    'Pan paniscus':     'Bonobo',
    'Pongo pygmaeus':   'Orangutan',
    'Pongo abelii':     'Orangutan',
}

# Add common name
ape_ranges_proj['common_name'] = ape_ranges_proj['sci_name'].map(species_labels)

# Total range area per genus (in km²)
ape_ranges_proj['area_km2'] = ape_ranges_proj.geometry.area / 1_000_000
range_area = ape_ranges_proj.groupby('common_name')['area_km2'].sum()
print("Total range area (km²):")
print(range_area)

# Total protected area per genus (in km²)
protected_areas_proj['common_name'] = protected_areas_proj['sci_name'].map(species_labels)
protected_areas_proj['pa_area_km2'] = protected_areas_proj.geometry.area / 1_000_000
pa_area = protected_areas_proj.groupby('common_name')['pa_area_km2'].sum()
print("\nTotal protected area (km²):")
print(pa_area)

# Percentage protected
print("\nPercentage of range protected:")
print((pa_area / range_area * 100).round(1))

In [ ]:
# Summary table
import pandas as pd

summary = pd.DataFrame({
    'Range area (km²)': range_area,
    'Protected area (km²)': pa_area,
    'Protected (%)': (pa_area / range_area * 100).round(1)
})

print(summary)

In [ ]:
# Grand totals
total_range = range_area.sum()
total_pa = pa_area.sum()
total_pct = (total_pa / total_range * 100).round(1)

print(f"Total range area:     {total_range:,.0f} km²")
print(f"Total protected area: {total_pa:,.0f} km²")
print(f"Total protected:      {total_pct}%")

In [1]:
import os
os.environ['PROJ_DATA'] = r"C:\Users\yuan\AppData\Local\anaconda3\envs\geo"

import geopandas as gpd


In [3]:
import os
os.environ['PROJ_DATA'] = r"C:\Users\otmil\anaconda3\envs\geo\Library\share\proj"  # your actual path from before

import folium
from folium import GeoJson, LayerControl
import geopandas as gpd

ape_ranges = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Ape_ranges.shp')
protected_areas = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\PAs_within_ape_ranges.shp')
protected_areas = protected_areas.set_crs('EPSG:4326')

asia_species = ['Pongo pygmaeus', 'Pongo abelii', 'Pongo tapanuliensis']
pa_asia = protected_areas[protected_areas['sci_name'].isin(asia_species)]

print(f"pa_asia rows: {len(pa_asia)}")
print(protected_areas['sci_name'].unique())

C:\Users\yuan\AppData\Local\anaconda3\envs\geo\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect PROJ data files. Set PROJ_LIB environment variable to the correct path.
  _init_proj_data()


pa_asia rows: 162
<StringArray>
[    'Pan troglodytes',    'Gorilla beringei',     'Gorilla gorilla',
      'Pongo pygmaeus',        'Pongo abelii',        'Pan paniscus',
 'Pongo tapanuliensis']
Length: 7, dtype: str


In [4]:
import folium
from folium import GeoJson, LayerControl
import geopandas as gpd

#Import shape files
ape_ranges = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Ape_ranges.shp')
protected_areas = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\PAs_within_ape_ranges.shp')

# Fix protected areas CRS
protected_areas = protected_areas.set_crs('EPSG:4326')

# Filter to Asia
asia_species = ['Pongo pygmaeus', 'Pongo abelii', 'Pongo tapanuliensis']
ape_asia= ape_ranges[ape_ranges['sci_name'].isin(asia_species)]
ape_asia_dissolved = ape_asia.dissolve(by='sci_name').reset_index()
pa_asia = protected_areas[protected_areas['sci_name'].isin(asia_species)]

# Colour per species
species_colors = {
    'Pongo pygmaeus':    '#797d62',    
    'Pongo abelii':    '#ffcb69',  
    'Pongo tapanuliensis':       '#7dcfb6',   
}

species_labels = {
    'Pongo pygmaeus':    'Bornean Orangutan',
    'Pongo abelii':   'Sumatran Orangutan',
    'Pongo tapanuliensis':    'Tapanuli Orangutan'
}

# Build map
m_asia = folium.Map(location=[0, 112], zoom_start=6, tiles=None)

#Add basemap from Carto
folium.TileLayer('CartoDB positron', control=False).add_to(m_asia)

# Add one layer per species
for species in asia_species:
    species_data = ape_asia_dissolved[ape_asia_dissolved['sci_name'] == species]
    color = species_colors[species]
    label = species_labels[species]

    GeoJson(
        species_data,
        name=label,
        style_function=lambda x, c=color: {
            'fillColor': c,
            'color': c,
            'weight': 1,
            'fillOpacity': 0.5
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['sci_name'],
            aliases=['Species:']
        )
    ).add_to(m_asia)

# Protected areas layer
GeoJson(
    pa_asia,
    name='Protected Areas',
    style_function=lambda x: {
        'fillColor': '#64a6bd',
        'color': '#64a6bd',
        'weight': 1,
        'fillOpacity': 0.6
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAME_ENG', 'IUCN_CAT'],
        aliases=['Protected Area:', 'IUCN Category:']
    )
).add_to(m_asia)

LayerControl(collapsed=False, title = 'Select layers', hide_basemap=True).add_to(m_asia)

from folium.plugins import Fullscreen

Fullscreen(
    position='topleft',
    title='Full-screen',
    title_cancel='Exit full-screen',
    force_separate_button=True
).add_to(m_asia)

# Title
title_html = '''
<div style="position: fixed; top: 25px; left: 100px; z-index: 1000; 
background-color: white; 
padding: 10px; 
border-radius: 5px; 
box-shadow: 2px 2px 5px rgba(0,0,0,0.3); 
font-family: Arial;
font-size: 14px;">

'''
m_asia.get_root().html.add_child(folium.Element(title_html))

css_html = '''
<style>
.leaflet-top.leaflet-right {
    margin-top: 25px;
    margin-right: 100px;
}

.leaflet-control-layers {
    font-family: Arial;
    font-size: 14px;
    border-radius: 5px;
    border: 1px solid #cccccc;
    background-color: white;
}

.leaflet-control-layers-list::before {
    content: 'Layers';
    font-weight: bold;
    font-size: 13px;
    display: block;
    margin-bottom: 8px;
    padding-bottom: 6px;
    border-bottom: 1px solid #eeeeee;
}

.leaflet-control-layers-expanded {
    padding: 12px;
    width: 200px;
}

.leaflet-control-layers label {
    margin-bottom: 6px;
    display: flex;
    align-items: center;
    gap: 6px;
}

.leaflet-control-layers-overlays {
    font-size: 13px;
}

.leaflet-control-layers-toggle {
    background-size: 20px;
}
</style>
'''
m_asia.get_root().html.add_child(folium.Element(css_html))




m_asia.save('asia_map.html')
print("Asia map saved")

Asia map saved


In [5]:
import webbrowser
webbrowser.open('asia_map.html')

True

In [ ]:
import rasterio
print(rasterio.__version__)

In [ ]:
import rasterio

path = r'C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Hansen_Spatial\Hansen_GFC-2025-v1.13_lossyear_00N_020E.tif'

with rasterio.open(path) as src:
    print('CRS:', src.crs)
    print('Shape:', src.shape)
    print('Bounds:', src.bounds)
    print('dtype:', src.dtypes)

In [ ]:
path_tc = r'C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Hansen_Spatial\Hansen_GFC-2025-v1.13_treecover2000_00N_020E.tif'

with rasterio.open(path_tc) as src:
    print('CRS:', src.crs)
    print('Shape:', src.shape)
    print('Bounds:', src.bounds)
    print('dtype:', src.dtypes)
    

In [ ]:
import rasterio
import geopandas as gpd
import numpy as np
from shapely.geometry import box
from rasterio.mask import mask as rasterio_mask

# Load ape ranges
ape_ranges = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Ape_ranges.shp')

# Draw a rectangle matching the tile boundaries
tile_bounds = box(20, -10, 30, 0)

# Keep only ape ranges that overlap with that rectangle
ape_tile = ape_ranges[ape_ranges.intersects(tile_bounds)]

print(f'Species in this tile: {ape_tile["sci_name"].unique()}')
print(f'Number of polygons: {len(ape_tile)}')

In [ ]:
from rasterio.mask import mask as rasterio_mask
import numpy as np

lossyear_path = r'C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Hansen_Spatial\Hansen_GFC-2025-v1.13_lossyear_00N_020E.tif'

# Use ape range polygons as the mask
geometries = ape_tile.geometry.values

with rasterio.open(lossyear_path) as src:
    loss_clipped, loss_transform = rasterio_mask(src, geometries, crop=True, nodata=255)
    loss_profile = src.profile

loss_data = loss_clipped[0]  # First band

print('Clipped shape:', loss_data.shape)
print('Unique values (sample):', np.unique(loss_data[:100, :100]))

In [ ]:
# Exclude nodata pixels (255) and look at all values
valid_pixels = loss_data[loss_data != 255]

print('Total valid pixels:', len(valid_pixels))
print('Pixels with no loss (0):', np.sum(valid_pixels == 0))
print('Pixels with loss:', np.sum(valid_pixels > 0))
print('All unique values:', np.unique(valid_pixels))


In [ ]:
protected_areas = gpd.read_file(r'C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\PAs_within_ape_ranges.shp')
protected_areas = protected_areas.set_crs('EPSG:4326')

# Clip PAs to the tile boundary
pa_tile = protected_areas[protected_areas.intersects(tile_bounds)]

print(f'Number of PAs in this tile: {len(pa_tile)}')
print(f'PA species coverage: {pa_tile["sci_name"].unique()}')

In [ ]:
from rasterio.features import rasterize
from rasterio.transform import from_bounds

# Create a raster grid matching the clipped lossyear raster
pa_mask = rasterize(
    [(geom, 1) for geom in pa_tile.geometry],
    out_shape=loss_data.shape,
    transform=loss_transform,
    fill=0,
    dtype='uint8'
)

print('PA mask shape:', pa_mask.shape)
print('Pixels inside PAs:', np.sum(pa_mask == 1))
print('Pixels outside PAs:', np.sum(pa_mask == 0))

In [ ]:
# Valid pixels = inside ape ranges (not nodata)
valid = loss_data != 255

# Inside PA and within ape range
inside_pa = valid & (pa_mask == 1)

# Outside PA and within ape range
outside_pa = valid & (pa_mask == 0)

# Count loss pixels in each group (values 1-25 = loss)
loss_inside = np.sum((loss_data[inside_pa] > 0))
loss_outside = np.sum((loss_data[outside_pa] > 0))

# Total pixels in each group
total_inside = np.sum(inside_pa)
total_outside = np.sum(outside_pa)

# Express as percentage
pct_inside = (loss_inside / total_inside) * 100
pct_outside = (loss_outside / total_outside) * 100

print(f'Forest loss inside PAs:  {pct_inside:.2f}%')
print(f'Forest loss outside PAs: {pct_outside:.2f}%')

In [ ]:
treecover_path = r'C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Hansen_Spatial\Hansen_GFC-2025-v1.13_treecover2000_00N_020E.tif'

# Clip treecover2000 to the same ape range area
with rasterio.open(treecover_path) as src:
    tc_clipped, _ = rasterio_mask(src, geometries, crop=True, nodata=255)

tc_data = tc_clipped[0]

# Forested in 2000 = canopy cover >= 10% AND not nodata
forested_2000 = (tc_data >= 10) & (tc_data != 255)

print('Total forested pixels in 2000:', np.sum(forested_2000))
print('As % of valid pixels:', (np.sum(forested_2000) / np.sum(valid) * 100).round(1))



In [ ]:
# Valid pixels = inside ape ranges AND forested in 2000
valid_forest = valid & forested_2000

# Inside PA and forested in 2000
inside_pa_forest = valid_forest & (pa_mask == 1)

# Outside PA and forested in 2000
outside_pa_forest = valid_forest & (pa_mask == 0)

# Count loss pixels in each group
loss_inside = np.sum(loss_data[inside_pa_forest] > 0)
loss_outside = np.sum(loss_data[outside_pa_forest] > 0)

# Total forested pixels in each group
total_inside = np.sum(inside_pa_forest)
total_outside = np.sum(outside_pa_forest)

# Express as percentage
pct_inside = (loss_inside / total_inside) * 100
pct_outside = (loss_outside / total_outside) * 100

print(f'Forest loss inside PAs:  {pct_inside:.2f}%')
print(f'Forest loss outside PAs: {pct_outside:.2f}%')

In [ ]:
years = list(range(1, 26))  # 1-25 representing 2001-2025

results = []

for year in years:
    # Pixels that lost forest in this specific year
    loss_this_year = loss_data == year
    
    # Inside PA losses this year
    loss_inside_yr = np.sum(loss_this_year & inside_pa_forest)
    
    # Outside PA losses this year
    loss_outside_yr = np.sum(loss_this_year & outside_pa_forest)
    
    # Express as % of 2000 forest extent in each group
    pct_inside_yr = (loss_inside_yr / total_inside) * 100
    pct_outside_yr = (loss_outside_yr / total_outside) * 100
    
    results.append({
        'year': 2000 + year,
        'loss_pct_inside_pa': pct_inside_yr,
        'loss_pct_outside_pa': pct_outside_yr
    })

import pandas as pd
results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
import pandas as pd

df = pd.read_csv(r'C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Forest_data.csv', encoding='latin-1')

drc = df[(df['country'] == 'Democratic Republic of the Congo') & (df['threshold'] == 10)]

drc_2013 = drc['tc_loss_ha_2013'].values[0]
drc_extent = drc['extent_2000_ha'].values[0]

print(f'DRC 2013 loss: {drc_2013:,.0f} ha')
print(f'DRC 2000 extent: {drc_extent:,.0f} ha')
print(f'DRC 2013 loss %: {(drc_2013/drc_extent*100):.3f}%')

In [ ]:
print(ape_ranges.columns.tolist())

In [ ]:
def analyze_tile(lossyear_path, treecover_path, ape_ranges, protected_areas):
    
    with rasterio.open(lossyear_path) as src:
        bounds = src.bounds
        tile_bounds = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
        src_transform = src.transform
        src_crs = src.crs
    
    # Filter to tile
    ape_tile = ape_ranges[ape_ranges.intersects(tile_bounds)]
    if len(ape_tile) == 0:
        print(f'No ape ranges in tile — skipping')
        return None
    
    pa_tile = protected_areas[protected_areas.intersects(tile_bounds)]
    
    # Per-country accumulators
    country_data = {}
    
    for idx, ape_row in ape_tile.iterrows():
        country = ape_row['country']
        geom = ape_row.geometry
        geom_bounds = geom.bounds
        
        if country not in country_data:
            country_data[country] = {
                'year_loss_inside': np.zeros(25, dtype=np.int64),
                'year_loss_outside': np.zeros(25, dtype=np.int64),
                'total_inside': 0,
                'total_outside': 0
            }
        
        # Read just the window for this single polygon
        with rasterio.open(lossyear_path) as src:
            window = from_bounds(geom_bounds[0], geom_bounds[1], 
                               geom_bounds[2], geom_bounds[3], 
                               src.transform)
            loss_chunk = src.read(1, window=window)
            win_transform = src.window_transform(window)
        
        with rasterio.open(treecover_path) as src:
            window = from_bounds(geom_bounds[0], geom_bounds[1],
                               geom_bounds[2], geom_bounds[3],
                               src.transform)
            tc_chunk = src.read(1, window=window)
        
        # Masks for this polygon
        from rasterio.features import rasterize
        ape_mask = rasterize(
            [(geom, 1)],
            out_shape=loss_chunk.shape,
            transform=win_transform,
            fill=0,
            dtype='uint8'
        )
        
        # PAs intersecting this polygon
        pa_here = pa_tile[pa_tile.intersects(geom)]
        if len(pa_here) > 0:
            pa_mask = rasterize(
                [(g, 1) for g in pa_here.geometry],
                out_shape=loss_chunk.shape,
                transform=win_transform,
                fill=0,
                dtype='uint8'
            )
        else:
            pa_mask = np.zeros(loss_chunk.shape, dtype='uint8')
        
        # Valid forest pixels within this polygon
        valid_forest = (ape_mask == 1) & (tc_chunk >= 10) & (tc_chunk != 255) & (loss_chunk != 255)
        
        inside = valid_forest & (pa_mask == 1)
        outside = valid_forest & (pa_mask == 0)
        
        country_data[country]['total_inside'] += int(np.sum(inside))
        country_data[country]['total_outside'] += int(np.sum(outside))
        
        for year_idx in range(1, 26):
            country_data[country]['year_loss_inside'][year_idx-1] += np.count_nonzero((loss_chunk == year_idx) & inside)
            country_data[country]['year_loss_outside'][year_idx-1] += np.count_nonzero((loss_chunk == year_idx) & outside)
        
        del loss_chunk, tc_chunk, ape_mask, pa_mask, valid_forest, inside, outside
    
    # Build results
    results = []
    for country, data in country_data.items():
        for year_idx in range(25):
            pct_inside = (data['year_loss_inside'][year_idx] / data['total_inside'] * 100) if data['total_inside'] > 0 else 0
            pct_outside = (data['year_loss_outside'][year_idx] / data['total_outside'] * 100) if data['total_outside'] > 0 else 0
            results.append({
                'country': country,
                'year': 2001 + year_idx,
                'loss_pct_inside_pa': pct_inside,
                'loss_pct_outside_pa': pct_outside,
                'pixels_inside_pa': data['total_inside'],
                'pixels_outside_pa': data['total_outside']
            })
    
    print(f'Tile complete: {lossyear_path.split(chr(92))[-1]}')
    return pd.DataFrame(results)

In [ ]:
def analyze_tile(lossyear_path, treecover_path, ape_ranges, protected_areas):
    
    with rasterio.open(lossyear_path) as src:
        bounds = src.bounds
        tile_bounds = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
        src_transform = src.transform
        src_crs = src.crs
    
    # Filter to tile
    ape_tile = ape_ranges[ape_ranges.intersects(tile_bounds)]
    if len(ape_tile) == 0:
        print(f'No ape ranges in tile — skipping')
        return None
    
    pa_tile = protected_areas[protected_areas.intersects(tile_bounds)]
    
    year_loss_inside = np.zeros(25, dtype=np.int64)
    year_loss_outside = np.zeros(25, dtype=np.int64)
    total_inside = 0
    total_outside = 0
    
    # Process one species polygon at a time
    for idx, ape_row in ape_tile.iterrows():
        geom = ape_row.geometry
        geom_bounds = geom.bounds  # minx, miny, maxx, maxy
        
        # Read just the window for this single polygon
        with rasterio.open(lossyear_path) as src:
            window = from_bounds(geom_bounds[0], geom_bounds[1], 
                               geom_bounds[2], geom_bounds[3], 
                               src.transform)
            loss_chunk = src.read(1, window=window)
            win_transform = src.window_transform(window)
        
        with rasterio.open(treecover_path) as src:
            window = from_bounds(geom_bounds[0], geom_bounds[1],
                               geom_bounds[2], geom_bounds[3],
                               src.transform)
            tc_chunk = src.read(1, window=window)
        
        # Masks for this polygon
        from rasterio.features import rasterize
        ape_mask = rasterize(
            [(geom, 1)],
            out_shape=loss_chunk.shape,
            transform=win_transform,
            fill=0,
            dtype='uint8'
        )
        
        # PAs intersecting this polygon
        pa_here = pa_tile[pa_tile.intersects(geom)]
        if len(pa_here) > 0:
            pa_mask = rasterize(
                [(g, 1) for g in pa_here.geometry],
                out_shape=loss_chunk.shape,
                transform=win_transform,
                fill=0,
                dtype='uint8'
            )
        else:
            pa_mask = np.zeros(loss_chunk.shape, dtype='uint8')
        
        # Valid forest pixels within this polygon
        valid_forest = (ape_mask == 1) & (tc_chunk >= 10) & (tc_chunk != 255) & (loss_chunk != 255)
        
        inside = valid_forest & (pa_mask == 1)
        outside = valid_forest & (pa_mask == 0)
        
        total_inside += int(np.sum(inside))
        total_outside += int(np.sum(outside))
        
        # Count loss by year
        for year_idx in range(1, 26):
            year_loss_inside[year_idx-1] += np.count_nonzero((loss_chunk == year_idx) & inside)
            year_loss_outside[year_idx-1] += np.count_nonzero((loss_chunk == year_idx) & outside)
        
        del loss_chunk, tc_chunk, ape_mask, pa_mask, valid_forest, inside, outside
    
    # Build results
    results = []
    for year_idx in range(25):
        pct_inside = (year_loss_inside[year_idx] / total_inside * 100) if total_inside > 0 else 0
        pct_outside = (year_loss_outside[year_idx] / total_outside * 100) if total_outside > 0 else 0
        results.append({
            'year': 2001 + year_idx,
            'loss_pct_inside_pa': pct_inside,
            'loss_pct_outside_pa': pct_outside,
            'pixels_inside_pa': total_inside,
            'pixels_outside_pa': total_outside
        })
    
    print(f'Tile complete: {lossyear_path.split(chr(92))[-1]}')
    return pd.DataFrame(results)

In [ ]:
import os
import gc

hansen_folder = r'C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Hansen_Spatial'

# All tile names
tiles = [
    '00N_010E', '00N_020E', '00N_030E',
    '10N_000E', '10N_010E', '10N_020E',
    '10S_010E', '10S_020E', '10S_030E',
    '00N_100E', '00N_110E', '00N_120E',
    '10N_100E', '10N_110E',
    '10S_100E', '10S_110E'
]

# Build file paths for each tile pair
tile_pairs = []
for tile in tiles:
    lossyear = os.path.join(hansen_folder, f'Hansen_GFC-2025-v1.13_lossyear_{tile}.tif')
    treecover = os.path.join(hansen_folder, f'Hansen_GFC-2025-v1.13_treecover2000_{tile}.tif')
    tile_pairs.append((lossyear, treecover))

# Run pipeline across all tiles
all_results = []
for lossyear_path, treecover_path in tile_pairs:
    result = analyze_tile(lossyear_path, treecover_path, ape_ranges, protected_areas)
    if result is not None:
        all_results.append(result)
    gc.collect()

print(f'Processed {len(all_results)} tiles with ape ranges')

In [ ]:
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import box
from rasterio.mask import mask as rasterio_mask
from rasterio.windows import from_bounds
from rasterio.features import rasterize
import gc
import os

# Load shapefiles
ape_ranges = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Ape_ranges.shp')
protected_areas = gpd.read_file(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\PAs_within_ape_ranges.shp')
protected_areas = protected_areas.set_crs('EPSG:4326')

In [ ]:
def analyze_tile(lossyear_path, treecover_path, ape_ranges, protected_areas):
    
    with rasterio.open(lossyear_path) as src:
        bounds = src.bounds
        tile_bounds = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
        src_transform = src.transform
        src_crs = src.crs
    
    # Filter to tile
    ape_tile = ape_ranges[ape_ranges.intersects(tile_bounds)]
    if len(ape_tile) == 0:
        print(f'No ape ranges in tile — skipping')
        return None
    
    pa_tile = protected_areas[protected_areas.intersects(tile_bounds)]
    
    # Per-country accumulators
    country_data = {}
    
    for idx, ape_row in ape_tile.iterrows():
        country = ape_row['country']
        geom = ape_row.geometry
        geom_bounds = geom.bounds
        
        if country not in country_data:
            country_data[country] = {
                'year_loss_inside': np.zeros(25, dtype=np.int64),
                'year_loss_outside': np.zeros(25, dtype=np.int64),
                'total_inside': 0,
                'total_outside': 0
            }
        
        # Read just the window for this single polygon
        with rasterio.open(lossyear_path) as src:
            window = from_bounds(geom_bounds[0], geom_bounds[1], 
                               geom_bounds[2], geom_bounds[3], 
                               src.transform)
            loss_chunk = src.read(1, window=window)
            win_transform = src.window_transform(window)
        
        with rasterio.open(treecover_path) as src:
            window = from_bounds(geom_bounds[0], geom_bounds[1],
                               geom_bounds[2], geom_bounds[3],
                               src.transform)
            tc_chunk = src.read(1, window=window)
        
        # Masks for this polygon
        from rasterio.features import rasterize
        ape_mask = rasterize(
            [(geom, 1)],
            out_shape=loss_chunk.shape,
            transform=win_transform,
            fill=0,
            dtype='uint8'
        )
        
        # PAs intersecting this polygon
        pa_here = pa_tile[pa_tile.intersects(geom)]
        if len(pa_here) > 0:
            pa_mask = rasterize(
                [(g, 1) for g in pa_here.geometry],
                out_shape=loss_chunk.shape,
                transform=win_transform,
                fill=0,
                dtype='uint8'
            )
        else:
            pa_mask = np.zeros(loss_chunk.shape, dtype='uint8')
        
        # Valid forest pixels within this polygon
        valid_forest = (ape_mask == 1) & (tc_chunk >= 10) & (tc_chunk != 255) & (loss_chunk != 255)
        
        inside = valid_forest & (pa_mask == 1)
        outside = valid_forest & (pa_mask == 0)
        
        country_data[country]['total_inside'] += int(np.sum(inside))
        country_data[country]['total_outside'] += int(np.sum(outside))
        
        for year_idx in range(1, 26):
            country_data[country]['year_loss_inside'][year_idx-1] += np.count_nonzero((loss_chunk == year_idx) & inside)
            country_data[country]['year_loss_outside'][year_idx-1] += np.count_nonzero((loss_chunk == year_idx) & outside)
        
        del loss_chunk, tc_chunk, ape_mask, pa_mask, valid_forest, inside, outside
    
    # Build results
    results = []
    for country, data in country_data.items():
        for year_idx in range(25):
            pct_inside = (data['year_loss_inside'][year_idx] / data['total_inside'] * 100) if data['total_inside'] > 0 else 0
            pct_outside = (data['year_loss_outside'][year_idx] / data['total_outside'] * 100) if data['total_outside'] > 0 else 0
            results.append({
                'country': country,
                'year': 2001 + year_idx,
                'loss_pct_inside_pa': pct_inside,
                'loss_pct_outside_pa': pct_outside,
                'pixels_inside_pa': data['total_inside'],
                'pixels_outside_pa': data['total_outside']
            })
    
    print(f'Tile complete: {lossyear_path.split(chr(92))[-1]}')
    return pd.DataFrame(results)

In [ ]:
combined = pd.concat(all_results).groupby('year').agg({
    'loss_pct_inside_pa': 'mean',
    'loss_pct_outside_pa': 'mean'
}).reset_index()

print(combined)


In [ ]:
# Remove 10S_100E from tiles list
tiles = [
    '00N_010E', '00N_020E', '00N_030E',
    '10N_000E', '10N_010E', '10N_020E',
    '10S_010E', '10S_020E', '10S_030E',
    '00N_100E', '00N_110E', '00N_120E',
    '10N_100E', '10N_110E',
    '10S_110E'
]

# Rebuild tile pairs
tile_pairs = []
for tile in tiles:
    lossyear = os.path.join(hansen_folder, f'Hansen_GFC-2025-v1.13_lossyear_{tile}.tif')
    treecover = os.path.join(hansen_folder, f'Hansen_GFC-2025-v1.13_treecover2000_{tile}.tif')
    tile_pairs.append((lossyear, treecover))

# Run only remaining tiles - skip first 9
for lossyear_path, treecover_path in tile_pairs[9:]:
    result = analyze_tile(lossyear_path, treecover_path, ape_ranges, protected_areas)
    if result is not None:
        all_results.append(result)
    gc.collect()

print(f'Total tiles processed: {len(all_results)}')

In [ ]:
# Combine all tiles with pixel-weighted averages
all_combined = pd.concat(all_results)

# Group by year and calculate weighted averages
weighted = all_combined.groupby('year').apply(
    lambda x: pd.Series({
        'loss_pct_inside_pa': (x['loss_pct_inside_pa'] * x['pixels_inside_pa']).sum() / x['pixels_inside_pa'].sum(),
        'loss_pct_outside_pa': (x['loss_pct_outside_pa'] * x['pixels_outside_pa']).sum() / x['pixels_outside_pa'].sum()
    })
).reset_index()

print(weighted)

# Also print the totals
total_inside = all_combined.groupby('year')['pixels_inside_pa'].first().sum()
total_outside = all_combined.groupby('year')['pixels_outside_pa'].first().sum()
print(f'\nTotal forested pixels inside PAs: {total_inside:,}')
print(f'Total forested pixels outside PAs: {total_outside:,}')


In [ ]:
import os
os.environ['PROJ_LIB'] = r'C:\Users\yuan\AppData\Local\anaconda3\Library\share\proj'
os.environ['GDAL_DATA'] = r'C:\Users\yuan\AppData\Local\anaconda3\Library\share\gdal'

import geopandas as gpd

In [ ]:
weighted.to_csv(
    r'C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\pa_forest_loss_results.csv',
    index=False
)
print('Saved successfully')

In [ ]:
africa_tiles_list = [
    '00N_010E', '00N_020E', '00N_030E',
    '10N_000E', '10N_010E', '10N_020E',
    '10S_010E', '10S_020E', '10S_030E'
]

asia_tiles_list = [
    '00N_100E', '00N_110E', '00N_120E',
    '10N_100E', '10N_110E', '10S_110E'
]

# Tag each result with its region
tagged_results = []
for i, (lossyear_path, treecover_path) in enumerate(tile_pairs):
    tile_name = lossyear_path.split('\\')[-1]
    tile_id = tile_name.replace('Hansen_GFC-2025-v1.13_lossyear_', '').replace('.tif', '')
    
    if i < len(all_results):
        df = all_results[i].copy()
        if tile_id in africa_tiles_list:
            df['region'] = 'Africa'
        elif tile_id in asia_tiles_list:
            df['region'] = 'Asia'
        else:
            df['region'] = 'Unknown'
        tagged_results.append(df)

all_tagged = pd.concat(tagged_results)
print(all_tagged['region'].value_counts())

In [ ]:
regional = all_tagged.groupby(['region', 'year']).apply(
    lambda x: pd.Series({
        'loss_pct_inside_pa': (x['loss_pct_inside_pa'] * x['pixels_inside_pa']).sum() / x['pixels_inside_pa'].sum(),
        'loss_pct_outside_pa': (x['loss_pct_outside_pa'] * x['pixels_outside_pa']).sum() / x['pixels_outside_pa'].sum()
    }),
    include_groups=False
).reset_index()

print(regional)

In [ ]:
weighted.to_csv(
    r'C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\pa_forest_loss_results_regions.csv',
    index=False
)
print('Saved successfully')

In [ ]:
weighted.to_csv(
    r'C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\pa_forest_loss_global.csv',
    index=False
)

regional.to_csv(
    r'C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\pa_forest_loss_regional.csv',
    index=False
)

print('Both files saved')

In [ ]:
# Sum annual percentages for cumulative total
cumulative = weighted.agg({
    'loss_pct_inside_pa': 'sum',
    'loss_pct_outside_pa': 'sum'
})
print('Global cumulative:')
print(cumulative)

cumulative_regional = regional.groupby('region').agg({
    'loss_pct_inside_pa': 'sum',
    'loss_pct_outside_pa': 'sum'
})
print('\nRegional cumulative:')
print(cumulative_regional)

In [ ]:
import pandas as pd
import plotly.graph_objects as go

global_df = pd.read_csv(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\pa_forest_loss_global.csv')
regional_df = pd.read_csv(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\pa_forest_loss_regional.csv')

# Cumulative totals
global_cumulative = global_df[['loss_pct_inside_pa', 'loss_pct_outside_pa']].sum()
regional_cumulative = regional_df.groupby('region')[['loss_pct_inside_pa', 'loss_pct_outside_pa']].sum()

print('Global cumulative:')
print(global_cumulative)
print('\nRegional cumulative:')
print(regional_cumulative)

In [ ]:
import pandas as pd
import plotly.graph_objects as go

global_df = pd.read_csv(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\pa_forest_loss_global.csv')
regional_df = pd.read_csv(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\pa_forest_loss_regional.csv')

# Cumulative totals
global_cumulative = global_df[['loss_pct_inside_pa', 'loss_pct_outside_pa']].sum()
regional_cumulative = regional_df.groupby('region')[['loss_pct_inside_pa', 'loss_pct_outside_pa']].sum()

fig = go.Figure()

# Africa inside
fig.add_trace(go.Bar(
    name='Inside Protected Areas',
    x=['Africa', 'Asia'],
    y=[regional_cumulative.loc['Africa', 'loss_pct_inside_pa'],
       regional_cumulative.loc['Asia', 'loss_pct_inside_pa']],
    marker_color='#2d6a4f',
    offsetgroup=0
))

# Africa outside
fig.add_trace(go.Bar(
    name='Outside Protected Areas',
    x=['Africa', 'Asia'],
    y=[regional_cumulative.loc['Africa', 'loss_pct_outside_pa'],
       regional_cumulative.loc['Asia', 'loss_pct_outside_pa']],
    marker_color='#7b2d42',
    offsetgroup=1
))

fig.update_layout(
    title='<b>Cumulative forest loss inside vs outside protected areas, 2001–2025</b><br><sup><i>Expressed as % of 2000 forest extent within great ape ranges</i></sup>',
    barmode='group',
    template='simple_white',
    font_family='Roboto',
    title_font_size=18,
    title_x=0.02,
    height=500,
    yaxis=dict(
        title='Cumulative forest cover loss (%)',
        gridcolor='lightgrey',
        griddash='dash',
        gridwidth=0.5
    ),
    legend=dict(
        font=dict(size=12),
        title=dict(text='<b>Protection Status</b>', font=dict(size=12)),
	orientation = 'h',
	x=0.1,
	y=-0.12
    ),
    margin=dict(t=80, b=130)
)

fig.update_traces(
    hovertemplate='<b>%{x}</b><br>Cumulative loss: %{y:.2f}%<extra></extra>'
)

fig.add_annotation(
    text="<b>Data source</b>: Hansen et al. (2013), Global Forest Change v GFC-2025-v1.13; UNEP-WCMC and IUCN (2026), Protected Planet: The World Database on Protected Areas (WDPA)",
    xref='paper', yref='paper',
    x=-0.07, y=-0.25,
    showarrow=False,
    font=dict(size=12, color='grey', style='italic'),
    xanchor='left'
)

In [ ]:
import plotly.graph_objects as go
import pandas as pd

regional_df = pd.read_csv(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\pa_forest_loss_regional.csv')

africa = regional_df[regional_df['region'] == 'Africa']
asia = regional_df[regional_df['region'] == 'Asia']

fig2 = go.Figure()

# Africa inside
fig2.add_trace(go.Scatter(
    x=africa['year'],
    y=africa['loss_pct_inside_pa'],
    name='<b>Africa — Inside PAs</b>',
    mode='lines',
    line=dict(color='#2d6a4f', width=2),
    hovertemplate='%{y:.2f}%'
))

# Africa outside
fig2.add_trace(go.Scatter(
    x=africa['year'],
    y=africa['loss_pct_outside_pa'],
    name='<b>Africa — Outside PAs</b>',
    mode='lines',
    line=dict(color='#2d6a4f', width=2, dash='dot'),
    hovertemplate='%{y:.2f}%'
))

# Asia inside
fig2.add_trace(go.Scatter(
    x=asia['year'],
    y=asia['loss_pct_inside_pa'],
    name='<b>Asia — Inside PAs</b>',
    mode='lines',
    line=dict(color='#7b2d42', width=2),
    hovertemplate='%{y:.2f}%'
))

# Asia outside
fig2.add_trace(go.Scatter(
    x=asia['year'],
    y=asia['loss_pct_outside_pa'],
    name='<b>Asia — Outside PAs</b>',
    mode='lines',
    line=dict(color='#7b2d42', width=2, dash='dot'),
    hovertemplate='%{y:.2f}%'
))

fig2.update_layout(
    title='<b>Annual forest loss inside vs outside protected areas, 2001–2025</b><br><sup><i>Expressed as % of 2000 forest extent within great ape ranges</i></sup>',
    template='simple_white',
    font_family='Roboto',
    title_font_size=18,
    title_x=0.02,
    height=550,
    yaxis=dict(
        title='Annual forest cover loss (%)',
        gridcolor='lightgrey',
        griddash='dash',
        gridwidth=0.5
    ),
    xaxis=dict(range=[2001, 2025]),
    legend=dict(
        font=dict(size=13),
        orientation='h',
        x=0.5,
        y=-0.15,
        xanchor='center'
    ),
    margin=dict(t=80, b=150),
    hovermode='x unified',
    modebar_remove=['pan', 'select', 'lasso']
)

fig2.add_annotation(
    text="<b>Data source</b>: Hansen et al. (2013), Global Forest Change v GFC-2025-v1.13; UNEP-WCMC and IUCN (2026), Protected Planet: The World Database on Protected Areas (WDPA)",
    xref='paper', yref='paper',
    x=0, y=-0.38,
    showarrow=False,
    font=dict(size=12, color='grey', style='italic'),
    xanchor='left',
    align='left'
)

fig2.show()

In [ ]:
import pandas as pd
import plotly.graph_objects as go

regional_df = pd.read_csv(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\pa_forest_loss_regional.csv')
regional_cumulative = regional_df.groupby('region')[['loss_pct_inside_pa', 'loss_pct_outside_pa']].sum()
africa = regional_df[regional_df['region'] == 'Africa']
asia = regional_df[regional_df['region'] == 'Asia']

fig = go.Figure()

# BAR TRACES
fig.add_trace(go.Bar(
    name='<b>Inside Protected Areas</b>',
    x=['<b>Africa</b>', '<b>Asia</b>'],
    y=[regional_cumulative.loc['Africa', 'loss_pct_inside_pa'],
       regional_cumulative.loc['Asia', 'loss_pct_inside_pa']],
    marker_color='#2d6a4f',
    offsetgroup=0,
    visible=True
))

fig.add_trace(go.Bar(
    name='<b>Outside Protected Areas</b>',
    x=['<b>Africa</b>', '<b>Asia</b>'],
    y=[regional_cumulative.loc['Africa', 'loss_pct_outside_pa'],
       regional_cumulative.loc['Asia', 'loss_pct_outside_pa']],
    marker_color='#7b2d42',
    offsetgroup=1,
    visible=True
))

# LINE TRACES
fig.add_trace(go.Scatter(
    x=africa['year'], y=africa['loss_pct_inside_pa'],
    name='<b>Africa — Inside PAs</b>',
    mode='lines', line=dict(color='#2d6a4f', width=3),
    hovertemplate='%{y:.2f}%',
    visible=False
))

fig.add_trace(go.Scatter(
    x=africa['year'], y=africa['loss_pct_outside_pa'],
    name='<b>Africa — Outside PAs</b>',
    mode='lines', line=dict(color='#2d6a4f', width=2, dash='dot'),
    hovertemplate='%{y:.2f}%',
    visible=False
))

fig.add_trace(go.Scatter(
    x=asia['year'], y=asia['loss_pct_inside_pa'],
    name='<b>Asia — Inside PAs</b>',
    mode='lines', line=dict(color='#7b2d42', width=3),
    hovertemplate='%{y:.2f}%',
    visible=False
))

fig.add_trace(go.Scatter(
    x=asia['year'], y=asia['loss_pct_outside_pa'],
    name='<b>Asia — Outside PAs</b>',
    mode='lines', line=dict(color='#7b2d42', width=2, dash='dot'),
    hovertemplate='%{y:.2f}%',
    visible=False
))

fig.update_layout(
    title='<b>Forest loss inside vs outside protected areas, 2001–2025</b><br><sup><i>Expressed as % of 2000 forest extent within great ape ranges</i></sup>',
    template='simple_white',
    font_family='Roboto',
    title_font_size=18,
    title_x=0.02,
    height=550,
    barmode='group',
    yaxis=dict(
        title='Forest cover loss (%)',
        gridcolor='lightgrey',
        griddash='dash',
        gridwidth=0.5
    ),
    xaxis=dict(
        tickfont=dict(size=14),
        tickmode='array',
        tickvals=[2001, 2005, 2010, 2015, 2020, 2025]
    ),
    legend=dict(
        font=dict(size=13),
        orientation='h',
        x=0.5,
        y=-0.15,
        xanchor='center'
    ),
    margin=dict(t=100, b=160),
    hovermode='closest',
    modebar_remove=['pan2d', 'select2d', 'lasso2d', 'zoom2d'],
    updatemenus=[
        dict(
            type='buttons',
            direction='right',
            x=0.97,
            y=1.2,
            xanchor='right',
            bgcolor='white',
            bordercolor='#18bc9c',
            borderwidth=1,
            buttons=[
                dict(
                    label='Cumulative total',
                    method='update',
                    args=[
                        {'visible': [True, True, False, False, False, False]},
                        {'yaxis.title.text': 'Cumulative forest cover loss (%)'}
                    ]
                ),
                dict(
                    label='Trend over time',
                    method='update',
                    args=[
                        {'visible': [False, False, True, True, True, True]},
                        {'yaxis.title.text': 'Annual forest cover loss (%)'}
                    ]
                )
            ]
        )
    ]
)

fig.update_traces(
    selector=dict(type='bar'),
    hovertemplate='<b>%{x}</b><br>Cumulative loss: %{y:.2f}%<extra></extra>'
)

fig.add_annotation(
    text="<b>Data source</b>: Hansen et al. (2013), Global Forest Change v GFC-2025-v1.13; UNEP-WCMC and IUCN (2026), Protected Planet: The World Database on Protected Areas (WDPA)",
    xref='paper', yref='paper',
    x=0, y=-0.38,
    showarrow=False,
    font=dict(size=12, color='grey', style='italic'),
    xanchor='left',
    align='left'
)

fig.show()

In [ ]:
print(df[df['country'].str.contains('Congo', case=False)]['country'].unique())

In [ ]:
# ISO code mapping
iso_map = {
    'Malaysia': 'MYS',
    'Indonesia': 'IDN',
    'Nigeria': 'NGA',
    'Democratic Republic of the Congo': 'COD',
    'Tanzania': 'TZA',
    'Uganda': 'UGA',
    'Cameroon': 'CMR',
    'Rwanda': 'RWA',
    'Republic of the Congo': 'COG',
    'Burundi': 'BDI',
    'Gabon': 'GAB',
    'Central African Republic': 'CAF'
}

# Calculate cumulative loss per country
cumulative = df_long.groupby('country').apply(
    lambda x: (x['forest_loss_ha'].sum() / x['extent_2000_ha'].iloc[0]) * 100
).reset_index()
cumulative.columns = ['country', 'cumulative_loss']
cumulative['iso'] = cumulative['country'].map(iso_map)

print(cumulative)

In [ ]:
print(cumulative[cumulative['country'] == 'Nigeria'])

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# --- HEATMAP DATA ---
df = pd.read_csv(r'C:\Users\yuan\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project\Forest_data.csv', encoding='latin-1')

ape_countries = [
    'Democratic Republic of the Congo', 'Cameroon', 'Gabon',
    'Republic of the Congo', 'Central African Republic', 'Nigeria',
    'Uganda', 'Rwanda', 'Burundi', 'Tanzania',
    'Indonesia', 'Malaysia'
]

df_filtered = df[(df['threshold'] == 10) & (df['country'].isin(ape_countries))]
loss_columns = [col for col in df.columns if col.startswith('tc_loss_ha_')]
df_loss = df_filtered[['country', 'extent_2000_ha'] + loss_columns]

df_long = df_loss.melt(
    id_vars=['country', 'extent_2000_ha'],
    value_vars=loss_columns,
    var_name='year',
    value_name='forest_loss_ha'
)
df_long['year'] = df_long['year'].str.replace('tc_loss_ha_', '').astype(int)
df_long['loss_percent'] = (df_long['forest_loss_ha'] / df_long['extent_2000_ha']) * 100

df_heatmap = df_long.pivot(index='country', columns='year', values='loss_percent')

country_names = {
    'Democratic Republic of the Congo': 'DR Congo',
    'Republic of the Congo': 'Rep. of Congo',
    'Central African Republic': 'Central African Rep.'
}
df_heatmap = df_heatmap.rename(index=country_names)
df_heatmap['total'] = df_heatmap.sum(axis=1)
df_heatmap = df_heatmap.sort_values('total', ascending=True).drop(columns='total')

# --- CHOROPLETH DATA ---
iso_map = {
    'Malaysia': 'MYS',
    'Indonesia': 'IDN',
    'Nigeria': 'NGA',
    'Democratic Republic of the Congo': 'COD',
    'Tanzania': 'TZA',
    'Uganda': 'UGA',
    'Cameroon': 'CMR',
    'Rwanda': 'RWA',
    'Republic of the Congo': 'COG',
    'Burundi': 'BDI',
    'Gabon': 'GAB',
    'Central African Republic': 'CAF'
}

cumulative = df_long.groupby('country').apply(
    lambda x: (x['forest_loss_ha'].sum() / x['extent_2000_ha'].iloc[0]) * 100,
    include_groups=False
).reset_index()
cumulative.columns = ['country', 'cumulative_loss']
cumulative['iso'] = cumulative['country'].map(iso_map)
cumulative['country_short'] = cumulative['country'].map(country_names).fillna(cumulative['country'])

print('Data ready')

In [ ]:
fig_combined = go.Figure()

# --- HEATMAP TRACE ---
fig_combined.add_trace(go.Heatmap(
    z=df_heatmap.values,
    x=df_heatmap.columns.tolist(),
    y=df_heatmap.index.tolist(),
    xgap=1,
    ygap=1,
    zmax=1.2,
    zmin=0,
    zauto=False,
    colorscale='RdYlGn_r',
    hovertemplate='<b>%{y}</b><br>Year: %{x}<br>Forest loss: %{z:.3f}%<extra></extra>',
    colorbar=dict(
        title=dict(text='<b>Annual forest<br>loss (%)</b>', font=dict(size=12)),
        tickfont=dict(size=11),
        orientation='h',
        x=0.5,
        y=-0.35,
        xanchor='center',
        thickness=20,
        len=0.6
    ),
    visible=True
))

# --- CHOROPLETH TRACE ---
fig_combined.add_trace(go.Choropleth(
    locations=cumulative['iso'],
    z=cumulative['cumulative_loss'],
    text=cumulative['country_short'],
    colorscale='RdYlGn_r',
    zmin=0,
    zmax=25,
    zauto=False,
    marker_line_color='white',
    marker_line_width=0.5,
    colorbar=dict(
        title=dict(text='<b>Cumulative forest <br>loss (%)</b>', font=dict(size=12)),
        tickfont=dict(size=11),
        orientation='h',
        x=0.5,
        y=-0.35,
        xanchor='center',
        thickness=20,
        len=0.6
    ),
    hovertemplate='<b>%{text}</b><br>Cumulative forest loss: %{z:.1f}%<extra></extra>',
    visible=False
))

fig_combined.update_layout(
    title='<b>Forest loss by country, 2001–2025</b><br><sup><i>Expressed as % of 2000 forest extent within great ape range countries</i></sup>',
    template='simple_white',
    font_family='Roboto',
    title_font_size=18,
    title_x=0.02,
    height=550,
    geo=dict(
        showframe=False,
        showcoastlines=True,
        coastlinecolor='lightgrey',
        showland=True,
        landcolor='#f5f5f5',
        showocean=True,
        oceancolor='#e8f4f8',
        projection_type='natural earth',
        fitbounds='locations',
        bgcolor='rgba(0,0,0,0)',
        domain=dict(x=[0, 0.01], y=[0, 0.01]),
        lataxis=dict(range=[-20, 20]),
        lonaxis=dict(range=[-20, 145])
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=[2001, 2005, 2010, 2015, 2020, 2025],
        tickfont=dict(size=13),
        visible=True
    ),
    yaxis=dict(
        tickfont=dict(size=13),
        visible=True
    ),
    margin=dict(t=100, b=180, l=150),
    modebar_remove=['pan2d', 'select2d', 'lasso2d', 'zoom2d'],
    updatemenus=[
        dict(
            type='buttons',
            direction='right',
            x=1.0,
            y=1.2,
            xanchor='right',
            bgcolor='white',
            bordercolor='#dee2e6',
            borderwidth=1,
            font=dict(size=13, family='Roboto'),
            buttons=[
                dict(
                    label='Annual heatmap',
                    method='update',
                    args=[
                        {'visible': [True, False]},
                        {
                            'xaxis.visible': True,
                            'yaxis.visible': True,
                            'geo.domain.x': [0, 0.01],
                            'geo.domain.y': [0, 0.01]
                        }
                    ]
                ),
                dict(
                    label='Cumulative map',
                    method='update',
                    args=[
                        {'visible': [False, True]},
                        {
                            'xaxis.visible': False,
                            'yaxis.visible': False,
                            'geo.domain.x': [0, 1],
                            'geo.domain.y': [0, 1],
                            'geo.fitbounds': 'locations'
                        }
                    ]
                )
            ]
        )
    ]
)

fig_combined.add_annotation(
    text="<b>Data source</b>: Hansen et al. (2013), Global Forest Change v GFC-2025-v1.13; Global Forest Watch",
    xref='paper', yref='paper',
    x=-0.07, y=-0.42,
    showarrow=False,
    font=dict(size=12, color='#555555', style='italic'),
    xanchor='left',
    align='left'
)

fig_combined.show()

In [ ]:
fig_heat = go.Figure()

fig_heat.add_trace(go.Heatmap(
    z=df_heatmap.values,
    x=df_heatmap.columns.tolist(),
    y=df_heatmap.index.tolist(),
    xgap=1,
    ygap=1,
    zmax=1.2,
    zmin=0,
    zauto=False,
    colorscale='RdYlGn_r',
    hovertemplate='<b>%{y}</b><br>Year: %{x}<br>Forest loss: %{z:.3f}%<extra></extra>',
    colorbar=dict(
        title=dict(text='<b>Annual forest<br>loss (%)</b>', font=dict(size=12)),
        tickfont=dict(size=12),
        orientation='h',
        x=0.5,
        y=-0.3,
        xanchor='center',
        thickness=30,
        len=0.6
    )
))

fig_heat.update_layout(
    title=dict(
        text='<b>Forest loss by country, 2001–2025</b><br>'
             '<sup><i>Annual loss as % of 2000 forest extent</i></sup>',
        x=0.02
    ),
    template='simple_white',
    font_family='Roboto',
    height=600,
    xaxis=dict(
        tickmode='array',
        tickvals=[2001, 2005, 2010, 2015, 2020, 2025],
        tickfont=dict(size=13)
    ),
    yaxis=dict(tickfont=dict(size=13)),
    margin=dict(t=70, b=140),
    modebar_remove=['pan2d', 'select2d', 'lasso2d', 'zoom2d', 'zoomIn2d', 'zoomOut2d', 'autoScale2d', 'resetScale2d', 'toImage']
)

fig_heat.add_annotation(
    text="<b>Data source</b>: Hansen et al. (2013), Global Forest Change v GFC-2025-v1.13; Global Forest Watch",
    xref='paper', yref='paper',
    x=-0.07, y=-0.34,
    showarrow=False,
    font=dict(size=12, color='#555555', style='italic'),
    xanchor='left',
    align='left'
)


In [ ]:
fig_bar = go.Figure()

fig_bar.add_trace(go.Bar(
    x=cumulative["cumulative_loss"],
    y=cumulative["country_short"],
    orientation='h',
    marker=dict(
        color=cumulative["cumulative_loss"],
        colorscale='RdYlGn_r',
        cmin=0,
        cmax=25,
        showscale=False
    ),
    hovertemplate='<b>%{y}</b><br>Cumulative forest loss: %{x:.1f}%<extra></extra>'
))

fig_bar.update_layout(
    title=dict(
        text='<b>Forest loss by country, 2001–2025</b><br>'
             '<sup><i>Cumulative loss as % of 2000 forest extent</i></sup>',
        x=0.02
    ),
    template='simple_white',
    font_family='Roboto',
    height=600,
    margin=dict(t=35, b=100, l=20),
    xaxis=dict(
        title='Cumulative forest loss (%)',
        tickfont=dict(size=13)
    ),
    yaxis=dict(
        tickfont=dict(size=13),
        categoryorder='total ascending'
    ),
    modebar_remove=['pan2d', 'select2d', 'lasso2d', 'zoom2d', 'zoomIn2d',
                    'zoomOut2d', 'autoScale2d', 'resetScale2d', 'toImage']
)

fig_bar.add_annotation(
    text="<b>Data source</b>: Hansen et al. (2013), Global Forest Change v GFC-2025-v1.13; Global Forest Watch",
    xref='paper', yref='paper',
    x=-0.07, y=-0.18,
    showarrow=False,
    font=dict(size=12, color='#555555', style='italic'),
    xanchor='left',
    align='left'
)

In [ ]:
fig_combined = go.Figure()

# --- Heatmap trace ---
fig_combined.add_trace(go.Heatmap(
    z=df_heatmap.values,
    x=df_heatmap.columns.tolist(),
    y=df_heatmap.index.tolist(),
    xgap=1,
    ygap=1,
    zmax=1.2,
    zmin=0,
    zauto=False,
    colorscale='RdYlGn_r',
    showscale=False,
    hovertemplate='<b>%{y}</b><br>Year: %{x}<br>Forest loss: %{z:.3f}%<extra></extra>',
    visible=True,
    name='heatmap'
))

# --- Bar trace ---
fig_combined.add_trace(go.Bar(
    x=cumulative["cumulative_loss"],
    y=cumulative["country_short"],
    orientation='h',
    marker=dict(
        color=cumulative["cumulative_loss"],
        colorscale='RdYlGn_r',
        cmin=0,
        cmax=25,
        showscale=False
    ),
    hovertemplate='<b>%{y}</b><br>Cumulative forest loss: %{x:.1f}%<extra></extra>',
    visible=False,
    name='bar'
))

fig_combined.update_layout(
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0.02,
        y=1.08,
        xanchor='left',
        showactive=True,
        buttons=[
            dict(
                label='Annual heatmap',
                method='update',
                args=[
                    {'visible': [True, False]},
                    {
                        'xaxis.tickmode': 'array',
                        'xaxis.tickvals': [2001, 2005, 2010, 2015, 2020, 2025],
                        'xaxis.title.text': '',
                        'yaxis.categoryorder': 'trace'
                    }
                ]
            ),
            dict(
                label='Cumulative bar chart',
                method='update',
                args=[
                    {'visible': [False, True]},
                    {
                        'xaxis.tickmode': 'auto',
                        'xaxis.tickvals': [],
                        'xaxis.title.text': 'Cumulative forest loss (%)',
                        'yaxis.categoryorder': 'total ascending'
                    }
                ]
            )
        ]
    )],
    title=dict(
        text='<b>Forest loss by country, 2001–2025</b><br>'
             '<sup><i>Annual loss as % of 2000 forest extent</i></sup>',
        x=0.02
    ),
    template='simple_white',
    font_family='Roboto',
    height=600,
    margin=dict(t=80, b=80, l=20),
    xaxis=dict(
        tickmode='array',
        tickvals=[2001, 2005, 2010, 2015, 2020, 2025],
        tickfont=dict(size=13)
    ),
    yaxis=dict(
        tickfont=dict(size=13),
        categoryorder='trace'
    ),
    modebar_remove=['pan2d', 'select2d', 'lasso2d', 'zoom2d', 'zoomIn2d',
                    'zoomOut2d', 'autoScale2d', 'resetScale2d', 'toImage']
)

fig_combined.add_annotation(
    text="<b>Data source</b>: Hansen et al. (2013), Global Forest Change v GFC-2025-v1.13; Global Forest Watch",
    xref='paper', yref='paper',
    x=-0.07, y=-0.15,
    showarrow=False,
    font=dict(size=12, color='#555555', style='italic'),
    xanchor='left',
    align='left'
)

In [ ]:
import rasterio

# Use any one tile from your existing set
lossyear_path = 'r:\\Users\\OMILLER1\\OneDrive - United Nations\\Desktop\\Career\\Great Ape Conservation Project\\data\\Hansen_Spatial\\Hansen_GFC-2025-v1.13_lossyear_00N_010E.tif'
treecover_path = 'r:\\Users\\OMILLER1\\OneDrive - United Nations\\Desktop\\Career\\Great Ape Conservation Project\\data\\Hansen_Spatial\\Hansen_GFC-2025-v1.13_treecover2000_00N_010E.tif'

with rasterio.open(lossyear_path) as loss_src, rasterio.open(treecover_path) as tree_src:
    print("CRS match:      ", loss_src.crs == tree_src.crs)
    print("Transform match:", loss_src.transform == tree_src.transform)
    print("Shape match:     ", loss_src.shape == tree_src.shape)
    print()
    print("Loss CRS:  ", loss_src.crs)
    print("Tree CRS:  ", tree_src.crs)
    print("Loss nodata:", loss_src.nodata)
    print("Tree nodata:", tree_src.nodata)